## Levantamos el servicio con FastAPI

Para este script son necesarios los siguientes ficheros:

1.   requirements.txt
2.   modelo_xgb_replica.json


Nota: Este script fue realizado en Colab por lo que el path de lectura de ficheros incia en  "/content/", se recomienda verificar el path de ubicación si se trabaja en local.


In [1]:
# Instalar requirements con su versión
!pip install -r "/content/requirements.txt"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.5 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=068c1d51a2bd803f56ccfbdddf54ab928e1aa2a52cff95cbaade10ebee5674cb
  Stored in di

In [2]:
#Importar las librerias para FastAPI
from fastapi import FastAPI
import numpy as np
from pydantic import BaseModel
from xgboost import XGBRegressor
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Leo  nuestro modelo json
model = XGBRegressor()
model.load_model("/content/modelo_xgb_replica.json")

In [4]:
# definir las clases del diccionario de 1 alojamiento
app = FastAPI()

variables_cols = list(model.feature_names_in_)

class Alojamiento(BaseModel):
    accommodates: int
    bedrooms: int
    minimum_nights: int
    maximum_nights: int
    number_of_reviews_ltm: int
    availability_eoy: int
    review_scores_rating: float
    review_scores_location: float
    calculated_host_listings_count: int
    bathrooms_num: float
    antiguedad_como_host: int
    antiguedad_anuncio: int
    dist_center_km: float
    dist_beach_km: float
    dist_sagrada_familia_km: float
    dist_park_guell_km: float
    dist_camp_nou_km: float
    host_has_profile_pic_True: int
    room_type_Private_room: int
    room_type_Shared_room: int
    has_availability_True: int
    shared_bathroom_True: int
    es_gran_tenedor_True: int
    tiene_biografia_True: int
    licencia_estado_Licencia_registrada: int
    licencia_estado_Sin_dato: int
    neighbourhood_agrupado_Sant_Pere_Santa_Caterina_i_la_Ribera: int
    neighbourhood_agrupado_Sants: int
    neighbourhood_agrupado_el_Barri_Gòtic: int
    neighbourhood_agrupado_el_Fort_Pienc: int
    neighbourhood_agrupado_el_Poble_Sec: int
    neighbourhood_agrupado_el_Raval: int
    neighbourhood_agrupado_l_Antiga_Esquerra_de_l_Eixample: int
    neighbourhood_agrupado_la_Barceloneta: int
    neighbourhood_agrupado_la_Dreta_de_l_Eixample: int
    neighbourhood_agrupado_la_Sagrada_Família: int
    neighbourhood_agrupado_la_Vila_de_Gràcia: int
    sin_reviews_True: int
    host_location_agrupado_Desconocido: int
    property_type_agrupado_Entire_rental_unit: int
    property_type_agrupado_Otros: int
    property_type_agrupado_Private_room_in_rental_unit: int
    property_type_agrupado_Room_in_hotel: int
    tiene_descripcion_True: int
    has_ac_True: int
    has_elevator_True: int
    has_workstation_True: int
    has_dishwasher_True: int
    has_parking_True: int
    has_pool_True: int

# Definir la app con predict
@app.post("/predict")
def predict(alojamiento: Alojamiento):
    df = pd.DataFrame([alojamiento.dict()])[variables_cols]
    precio_pred_log = float(model.predict(df)[0])
    precio_pred = np.exp(precio_pred_log)
    return {"precio_pred": round(precio_pred, 2),
           "precio_pred_log": round(precio_pred_log, 2)}

In [5]:
#iniciar el servicio
import uvicorn

def run():
    uvicorn.run(app, host="0.0.0.0", port=4000, log_level="info")



In [6]:
import threading
thread = threading.Thread(target=run)

In [7]:
thread.start()

# Predecir precios de los alojamientos de Airbnb individualmente

### Nota: esta parte puede ser ejecutada en esta misma sesión o notebook o en otro notebook tanto en local o Colab.

Ejemplo 1.

1 persona, en 1 habitación, con 1 baño, 3 noches, en barrio Sants, con licencia, con elevador con un scores a partir de 4, anuncio tiene reviews y otras características de distancia y propiedad.

In [8]:
### Lanzo una petición al modelo

import requests

# Definir la URL del endpoint
url = 'http://localhost:4000/predict'
# Nueva data
data = {
    # Variables numéricas
    "accommodates": 1,
    "bedrooms": 1,
    "minimum_nights": 3,
    "maximum_nights": 3,
    "number_of_reviews_ltm": 5,
    "availability_eoy": 60,
    "review_scores_rating": 4,
    "review_scores_location": 4.9,
    "calculated_host_listings_count": 3,
    "bathrooms_num": 1,
    "antiguedad_como_host": 1,
    "antiguedad_anuncio": 1,
    "dist_center_km": 1.5,
    "dist_beach_km": 2,
    "dist_sagrada_familia_km": 1,
    "dist_park_guell_km": 3,
    "dist_camp_nou_km": 5,

    # Variables binarias (0/1)
    "host_has_profile_pic_True": 1,
    "room_type_Private_room": 0,
    "room_type_Shared_room": 0,
    "has_availability_True": 1,
    "shared_bathroom_True": 0,
    "es_gran_tenedor_True": 0,
    "tiene_biografia_True": 0,
    "licencia_estado_Licencia_registrada": 1,
    "licencia_estado_Sin_dato": 0,

    # Barrio (solo puede ser seleccionado 1 opcion)
    "neighbourhood_agrupado_Sant_Pere_Santa_Caterina_i_la_Ribera": 0,
    "neighbourhood_agrupado_Sants": 1,
    "neighbourhood_agrupado_el_Barri_Gòtic": 0,
    "neighbourhood_agrupado_el_Fort_Pienc": 0,
    "neighbourhood_agrupado_el_Poble_Sec": 0,
    "neighbourhood_agrupado_el_Raval": 0,
    "neighbourhood_agrupado_l_Antiga_Esquerra_de_l_Eixample": 0,
    "neighbourhood_agrupado_la_Barceloneta": 0,
    "neighbourhood_agrupado_la_Dreta_de_l_Eixample": 0,
    "neighbourhood_agrupado_la_Sagrada_Família": 0,
    "neighbourhood_agrupado_la_Vila_de_Gràcia": 0,

    "sin_reviews_True": 0,
    "host_location_agrupado_Desconocido": 0,

    # Tipo de propiedad (solo uno debe ser 1)
    "property_type_agrupado_Entire_rental_unit": 0,
    "property_type_agrupado_Otros": 0,
    "property_type_agrupado_Private_room_in_rental_unit": 0,
    "property_type_agrupado_Room_in_hotel": 0,
    #amenities
    "tiene_descripcion_True": 1,
    "has_ac_True": 0,
    "has_elevator_True": 1,
    "has_workstation_True": 0,
    "has_dishwasher_True": 0,
    "has_parking_True": 0,
    "has_pool_True": 0,
}


INFO:     Started server process [1550]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:4000 (Press CTRL+C to quit)


In [9]:
# Hacer la solicitud POST
response = requests.post(url, json=data)

# Imprimir la respuesta de prediccion
print(response.json())

INFO:     127.0.0.1:41398 - "POST /predict HTTP/1.1" 200 OK
{'precio_pred': 105.57, 'precio_pred_log': 4.66}


Ejemplo 2.

8 personas, en 5 habitaciones, 2 privadas y 3 compartidas en barrio con 5 baños, entre 5 a 7 días, en barrio la Barceloneta, con anuncio que incluye descripción, y otras características de distancias de centro y playa.


In [10]:
### Lanzo una petición al modelo

import requests

# Definir la URL del endpoint
url = 'http://localhost:4000/predict'
# Nueva data
data = {
    # Variables numéricas
    "accommodates": 8,
    "bedrooms": 5,
    "minimum_nights": 5,
    "maximum_nights": 7,
    "number_of_reviews_ltm": 0,
    "availability_eoy": 10,
    "review_scores_rating": 0,
    "review_scores_location": 0,
    "calculated_host_listings_count": 0,
    "bathrooms_num": 5,
    "antiguedad_como_host": 1,
    "antiguedad_anuncio": 1,
    "dist_center_km": 3,
    "dist_beach_km": 1,
    "dist_sagrada_familia_km": 1,
    "dist_park_guell_km": 1,
    "dist_camp_nou_km": 1,

    # Variables binarias (0/1)
    "host_has_profile_pic_True": 1,
    "room_type_Private_room": 2,
    "room_type_Shared_room": 3,
    "has_availability_True": 1,
    "shared_bathroom_True": 1,
    "es_gran_tenedor_True": 0,
    "tiene_biografia_True": 0,
    "licencia_estado_Licencia_registrada": 0,
    "licencia_estado_Sin_dato": 0,

    # Barrio (solo puede ser seleccionado 1 opcion)
    "neighbourhood_agrupado_Sant_Pere_Santa_Caterina_i_la_Ribera": 0,
    "neighbourhood_agrupado_Sants": 0,
    "neighbourhood_agrupado_el_Barri_Gòtic": 0,
    "neighbourhood_agrupado_el_Fort_Pienc": 0,
    "neighbourhood_agrupado_el_Poble_Sec": 0,
    "neighbourhood_agrupado_el_Raval": 0,
    "neighbourhood_agrupado_l_Antiga_Esquerra_de_l_Eixample": 0,
    "neighbourhood_agrupado_la_Barceloneta": 1,
    "neighbourhood_agrupado_la_Dreta_de_l_Eixample": 0,
    "neighbourhood_agrupado_la_Sagrada_Família": 0,
    "neighbourhood_agrupado_la_Vila_de_Gràcia": 0,

    "sin_reviews_True": 1,
    "host_location_agrupado_Desconocido": 0,

    # Tipo de propiedad (solo uno debe ser 1)
    "property_type_agrupado_Entire_rental_unit": 0,
    "property_type_agrupado_Otros": 0,
    "property_type_agrupado_Private_room_in_rental_unit": 1,
    "property_type_agrupado_Room_in_hotel": 0,
    # amenities
    "tiene_descripcion_True": 1,
    "has_ac_True": 0,
    "has_elevator_True": 0,
    "has_workstation_True": 0,
    "has_dishwasher_True": 0,
    "has_parking_True": 0,
    "has_pool_True": 0,
}


In [11]:
# Hacer la solicitud POST
response = requests.post(url, json=data)

# Imprimir la respuesta de prediccion
print(response.json())

INFO:     127.0.0.1:41412 - "POST /predict HTTP/1.1" 200 OK
{'precio_pred': 271.58, 'precio_pred_log': 5.6}
